In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!git clone https://github.com/eya-bouhmida/Multimodal-RAG-From-Scratch.git

import os, shutil
os.chdir('/content/Multimodal-RAG-From-Scratch')

if os.path.exists('data'):
    if os.path.islink('data'):
        os.unlink('data')
    else:
        shutil.rmtree('data')

os.symlink(
    '/content/drive/MyDrive/multimodal-rag-project/data',
    '/content/Multimodal-RAG-From-Scratch/data'
)
print(os.listdir('data/raw/'))

fatal: destination path 'Multimodal-RAG-From-Scratch' already exists and is not an empty directory.
['pubmed_multimodal', 'has_fr', 'who_en']


In [ ]:
!pip install -q "langchain-core==0.3.0" "langchain-community==0.3.0" "langchain-groq==0.2.0" ragas datasets

In [ ]:
!pip install sentence-transformers qdrant-client rank-bm25 groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 27.6 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from rank_bm25 import BM25Okapi
from groq import Groq
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2020/3890917367.py:10: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_2020/3890917367.py:10: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics impo

In [ ]:
import torch
!pip install python-dotenv -q

from dotenv import load_dotenv
import os

load_dotenv()
QDRANT_URL = "https://a52409e3-d81f-4182-a9f5-a23b1511daef.australia-southeast1-0.gcp.cloud.qdrant.io"
COLLECTION_NAME = "medlens"
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print(f"✅ Tout connecté sur {device}!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Tout connecté sur cuda!


In [ ]:
print("⏳ Chargement BM25...")

all_chunks = []
results, offset = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=5000,
    offset=None,
    with_payload=True,
    with_vectors=False
)
all_chunks.extend(results)

tokenized_chunks = [chunk.payload['text'].lower().split() for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"✅ BM25 initialisé avec {len(all_chunks)} chunks!")

⏳ Chargement BM25...
✅ BM25 initialisé avec 5000 chunks!


In [ ]:
def dense_search(query, top_k=20):
    query_vector = embedding_model.encode(query).tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True
    ).points
    return results

def bm25_search(query, top_k=20):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"id": all_chunks[idx].id, "score": scores[idx], "payload": all_chunks[idx].payload} for idx in top_indices]

def reciprocal_rank_fusion(dense_results, bm25_results, k=60):
    scores = {}
    for rank, result in enumerate(dense_results):
        doc_id = result.id
        if doc_id not in scores:
            scores[doc_id] = {"score": 0, "payload": result.payload}
        scores[doc_id]["score"] += 1 / (k + rank + 1)
    for rank, result in enumerate(bm25_results):
        doc_id = result["id"]
        if doc_id not in scores:
            scores[doc_id] = {"score": 0, "payload": result["payload"]}
        scores[doc_id]["score"] += 1 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True)[:20]

def rerank(query, fused_results, top_k=5):
    pairs = [[query, result[1]["payload"]["text"]] for result in fused_results]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, fused_results), key=lambda x: x[0], reverse=True)
    return ranked[:top_k]

def hybrid_search(query, top_k=5):
    dense_results = dense_search(query, top_k=20)
    bm25_results = bm25_search(query, top_k=20)
    fused = reciprocal_rank_fusion(dense_results, bm25_results)
    return rerank(query, fused, top_k=top_k)

def generate_answer(query):
    chunks = hybrid_search(query, top_k=5)
    context = ""
    contexts_list = []
    for score, result in chunks:
        text = result[1]["payload"]["text"]
        context += text + "\n"
        contexts_list.append(text)
    prompt = f"""Tu es MedLens, un assistant médical. Réponds uniquement en te basant sur les documents fournis. Cite tes sources.

DOCUMENTS:
{context}

QUESTION: {query}

RÉPONSE:"""
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.1
    )
    return response.choices[0].message.content, contexts_list

In [ ]:
# 20 questions médicales avec réponses attendues
eval_questions = [
    # English
    "What are the main symptoms of diabetes?",
    "What is hypertension and what causes it?",
    "What are the risk factors for cardiovascular disease?",
    "How is type 2 diabetes treated?",
    "What are the symptoms of depression?",
    "What is BMI and how is obesity defined?",
    "What vaccines are recommended for adults?",
    "What are the main causes of respiratory diseases?",
    "What is the normal blood pressure range?",
    "What are the symptoms of a heart attack?",
    # Français
    "Quels sont les symptômes du diabète?",
    "Comment traite-t-on l'hypertension artérielle?",
    "Quels sont les facteurs de risque du cancer?",
    "Qu'est-ce que la dépression et comment la traiter?",
    "Quels sont les effets de l'obésité sur la santé?",
    "Comment prévenir les maladies cardiovasculaires?",
    "Quels sont les symptômes d'une pneumonie?",
    "Qu'est-ce que le diabète de type 1?",
    "Comment fonctionne la vaccination?",
    "Quels sont les traitements de l'hypertension?"
]

ground_truths = [
    "Diabetes symptoms include increased thirst, frequent urination, fatigue, blurred vision, and slow healing wounds.",
    "Hypertension is high blood pressure, caused by factors including genetics, diet, stress, and lack of exercise.",
    "Risk factors include smoking, high blood pressure, high cholesterol, diabetes, obesity, and physical inactivity.",
    "Type 2 diabetes is treated with lifestyle changes, oral medications like metformin, and sometimes insulin.",
    "Depression symptoms include persistent sadness, loss of interest, fatigue, sleep problems, and difficulty concentrating.",
    "BMI is body mass index. Obesity is defined as BMI over 30.",
    "Adults need influenza, COVID-19, tetanus, and other vaccines based on age and risk factors.",
    "Respiratory diseases are caused by infections, smoking, air pollution, and allergies.",
    "Normal blood pressure is below 120/80 mmHg.",
    "Heart attack symptoms include chest pain, shortness of breath, sweating, and pain radiating to the arm.",
    "Les symptômes du diabète incluent soif excessive, urination fréquente, fatigue et vision floue.",
    "L'hypertension est traitée par des changements de mode de vie et des médicaments antihypertenseurs.",
    "Les facteurs de risque du cancer incluent le tabagisme, l'obésité, l'exposition aux rayonnements et les antécédents familiaux.",
    "La dépression est traitée par psychothérapie, antidépresseurs et changements de mode de vie.",
    "L'obésité augmente le risque de diabète, maladies cardiaques, hypertension et certains cancers.",
    "La prévention cardiovasculaire inclut exercice, alimentation saine, arrêt du tabac et contrôle de la tension.",
    "La pneumonie se manifeste par fièvre, toux, douleur thoracique et difficultés respiratoires.",
    "Le diabète de type 1 est une maladie auto-immune où le pancréas ne produit plus d'insuline.",
    "La vaccination stimule le système immunitaire pour produire des anticorps contre les maladies.",
    "Les traitements de l'hypertension incluent les diurétiques, bêtabloquants et inhibiteurs de l'ECA."
]

print(f"✅ {len(eval_questions)} questions d'évaluation prêtes!")

✅ 20 questions d'évaluation prêtes!


In [ ]:
print("⏳ Génération des réponses pour évaluation...")

questions = []
answers = []
contexts = []
truths = []

for i, (question, truth) in enumerate(zip(eval_questions, ground_truths)):
    try:
        answer, context_list = generate_answer(question)
        questions.append(question)
        answers.append(answer)
        contexts.append(context_list)
        truths.append(truth)
        print(f"  ✅ [{i+1}/{len(eval_questions)}] {question[:50]}...")
    except Exception as e:
        print(f"  ❌ [{i+1}/{len(eval_questions)}] Erreur: {e}")

print(f"\n✅ {len(answers)} réponses générées!")

⏳ Génération des réponses pour évaluation...
  ✅ [1/20] What are the main symptoms of diabetes?...
  ✅ [2/20] What is hypertension and what causes it?...
  ❌ [3/20] Erreur: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kx61m6ptfpzs40sbvtfze3z6` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 3847, Requested 2177. Please try again in 240ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  ✅ [4/20] How is type 2 diabetes treated?...
  ✅ [5/20] What are the symptoms of depression?...
  ✅ [6/20] What is BMI and how is obesity defined?...
  ✅ [7/20] What vaccines are recommended for adults?...
  ✅ [8/20] What are the main causes of respiratory diseases?...
  ✅ [9/20] What is the normal blood pressure range?...
  ✅ [10/20] What are the symptoms of a heart attack?...
  ✅ [11/20] Quels sont les symptômes du dia

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

eval_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def compute_faithfulness(answer, contexts):
    """Mesure si la réponse est fidèle aux contextes"""
    answer_emb = eval_model.encode(answer)
    context_embs = eval_model.encode(contexts)
    scores = util.cos_sim(answer_emb, context_embs)
    return float(scores.max())

def compute_answer_relevancy(question, answer):
    """Mesure si la réponse répond à la question"""
    q_emb = eval_model.encode(question)
    a_emb = eval_model.encode(answer)
    return float(util.cos_sim(q_emb, a_emb))

def compute_context_precision(answer, contexts):
    """Mesure si les contextes récupérés sont utiles"""
    answer_emb = eval_model.encode(answer)
    scores = []
    for ctx in contexts:
        ctx_emb = eval_model.encode(ctx)
        scores.append(float(util.cos_sim(answer_emb, ctx_emb)))
    return np.mean(scores)

# Évaluer toutes les questions
print("⏳ Évaluation en cours...")

faithfulness_scores = []
relevancy_scores = []
precision_scores = []

for i, (q, a, ctx, gt) in enumerate(zip(questions, answers, contexts, ground_truths)):
    f = compute_faithfulness(a, ctx)
    r = compute_answer_relevancy(q, a)
    p = compute_context_precision(a, ctx)

    faithfulness_scores.append(f)
    relevancy_scores.append(r)
    precision_scores.append(p)

    print(f"  [{i+1}/{len(questions)}] F:{f:.2f} R:{r:.2f} P:{p:.2f}")

print(f"\n🎉 RÉSULTATS MEDLENS:")
print(f"  Faithfulness:      {np.mean(faithfulness_scores):.3f}")
print(f"  Answer Relevancy:  {np.mean(relevancy_scores):.3f}")
print(f"  Context Precision: {np.mean(precision_scores):.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⏳ Évaluation en cours...
  [1/19] F:0.73 R:0.60 P:0.60
  [2/19] F:0.70 R:0.57 P:0.51
  [3/19] F:0.48 R:0.47 P:0.47
  [4/19] F:0.60 R:0.51 P:0.52
  [5/19] F:0.67 R:0.59 P:0.66
  [6/19] F:0.65 R:0.52 P:0.55
  [7/19] F:0.45 R:0.40 P:0.39
  [8/19] F:0.43 R:0.45 P:0.36
  [9/19] F:0.65 R:0.52 P:0.58
  [10/19] F:0.81 R:0.74 P:0.69
  [11/19] F:0.80 R:0.71 P:0.74
  [12/19] F:0.81 R:0.69 P:0.69
  [13/19] F:0.78 R:0.72 P:0.67
  [14/19] F:0.79 R:0.73 P:0.70
  [15/19] F:0.86 R:0.72 P:0.70
  [16/19] F:0.83 R:0.72 P:0.68
  [17/19] F:0.91 R:0.74 P:0.76
  [18/19] F:0.62 R:0.53 P:0.54
  [19/19] F:0.75 R:0.60 P:0.62

🎉 RÉSULTATS MEDLENS:
  Faithfulness:      0.701
  Answer Relevancy:  0.607
  Context Precision: 0.602


In [ ]:
import json
import numpy as np

# Calculer les moyennes depuis les listes de scores
results_dict = {
    "faithfulness": float(np.mean(faithfulness_scores)),
    "answer_relevancy": float(np.mean(relevancy_scores)),
    "context_precision": float(np.mean(precision_scores)),
    "num_questions": len(questions),
    "model": "llama-3.1-8b-instant",
    "embedding_model": "all-MiniLM-L6-v2"
}

with open('data/processed/ragas_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print("✅ Résultats sauvegardés!")
print(json.dumps(results_dict, indent=2))

✅ Résultats sauvegardés!
{
  "faithfulness": 0.7009162824404868,
  "answer_relevancy": 0.6074105171780837,
  "context_precision": 0.6017011786762037,
  "num_questions": 19,
  "model": "llama-3.1-8b-instant",
  "embedding_model": "all-MiniLM-L6-v2"
}
